# 分别使用男女足数据训练的 xA 模型

这个 notebook 仿照 enriched_xg_features_competition_events.ipynb：用 statsbombpy.sb.competition_events(...) 读取同一批男女足公开数据，分别训练男足和女足的 xA（expected assists）模型，然后让两个模型都给同一批 held-out 女足测试集传球打分。

主比较使用女足训练集规模下采样后的男足模型 vs 女足模型；男足 full-set 模型保留为补充诊断，用来观察训练数据量差异本身的影响。


## 0. 这里的 xA 怎么定义？

这里把 xA 定义成一次传球最终为球队制造的射门 xG 期望值。具体标签构造如下：

- 如果一脚传球在 StatsBomb 事件里有 assisted_shot_id，就把被助攻射门的 shot_statsbomb_xg 作为这脚传球的 realized xA label。
- 如果一脚传球没有直接助攻射门，label 就是 0。
- 模型输入使用传球本身的信息：起终点、长度、角度、是否传中 / 倒三角 / 直塞、是否受压、传球高度、身体部位、定位球上下文、传球结果、终点区域等。

因此它不是直接复制 StatsBomb 的 shot-assist 字段，而是在所有传球上训练一个 pass-level expected xA model；之后比较“男足训练出的 pass-to-shot-xG 关系”和“女足训练出的 pass-to-shot-xG 关系”在同一批女足测试传球上的差异。


## 1. Imports 和 constants

依赖应安装在项目 .venv 里。这个 notebook 不在运行时安装依赖，风格和已有 notebook 保持一致。


In [ ]:
import ast
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from mplsoccer import Pitch
import numpy as np
import pandas as pd
from scipy import stats
from statsbombpy import sb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
CACHE_DIR = Path("vaep_data")
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
WOMEN_TEST_SHARE = 0.30

MIN_PASSES_PER_PLAYER = 100
MIN_PASSES_PER_PASS_GROUP = 500
MIN_PLAYERS_FOR_BUCKET_SPEARMAN = 2

print("ready")


## 2. 比赛列表

和 xG / xT / VAEP notebook 使用同一批数据：女足 2023/24 top leagues + NWSL 2023，男足 2015/16 top leagues。


In [ ]:
WOMEN_COMPETITIONS = [
    {
        "competition_id": 37,
        "season_id": 281,
        "country": "England",
        "division": "FA Women's Super League",
        "season": "2023/2024",
        "gender": "female",
        "league": "WSL",
    },
    {
        "competition_id": 135,
        "season_id": 281,
        "country": "Germany",
        "division": "Frauen Bundesliga",
        "season": "2023/2024",
        "gender": "female",
        "league": "Frauen Bundesliga",
    },
    {
        "competition_id": 182,
        "season_id": 281,
        "country": "Spain",
        "division": "Liga F",
        "season": "2023/2024",
        "gender": "female",
        "league": "Liga F",
    },
    {
        "competition_id": 131,
        "season_id": 281,
        "country": "Italy",
        "division": "Serie A Women",
        "season": "2023/2024",
        "gender": "female",
        "league": "Serie A Women",
    },
    {
        "competition_id": 49,
        "season_id": 107,
        "country": "United States of America",
        "division": "NWSL",
        "season": "2023",
        "gender": "female",
        "league": "NWSL",
    },
]

MEN_COMPETITIONS = [
    {
        "competition_id": 11,
        "season_id": 27,
        "country": "Spain",
        "division": "La Liga",
        "season": "2015/2016",
        "gender": "male",
        "league": "La Liga",
    },
    {
        "competition_id": 2,
        "season_id": 27,
        "country": "England",
        "division": "Premier League",
        "season": "2015/2016",
        "gender": "male",
        "league": "Premier League",
    },
    {
        "competition_id": 12,
        "season_id": 27,
        "country": "Italy",
        "division": "Serie A",
        "season": "2015/2016",
        "gender": "male",
        "league": "Serie A",
    },
    {
        "competition_id": 7,
        "season_id": 27,
        "country": "France",
        "division": "Ligue 1",
        "season": "2015/2016",
        "gender": "male",
        "league": "Ligue 1",
    },
]


## 3. 加载比赛列表

仍然需要 match list，因为女足 train/test split 要在 game level 做，避免同一场比赛同时进入训练和测试。


In [ ]:
def load_remote_matches(competitions, cache_tag):
    """Load remote StatsBomb match lists for a set of competitions."""
    matches_cache = CACHE_DIR / f"{cache_tag}_matches_competition_events_method.pkl"
    if matches_cache.exists():
        print(f"[cache] {cache_tag} matches")
        return pd.read_pickle(matches_cache)

    match_frames = []
    for competition in competitions:
        try:
            matches = sb.matches(
                competition_id=competition["competition_id"],
                season_id=competition["season_id"],
            )
        except Exception as error:
            print("  !", competition["league"], error)
            continue

        matches = matches.copy()
        matches["league"] = competition["league"]
        match_frames.append(matches)
        print(f"[load matches] {competition['league']}: {len(matches)} games")

    all_matches = pd.concat(match_frames, ignore_index=True)
    all_matches.to_pickle(matches_cache)
    return all_matches


women_matches = load_remote_matches(WOMEN_COMPETITIONS, "women")
men_matches = load_remote_matches(MEN_COMPETITIONS, "men2015")

print("women games", len(women_matches), "men games", len(men_matches))


## 4. 共享设置：女足 game-level split

后续所有 xA 比较都只在同一批女足测试比赛上进行。男足模型、女足模型、男足 full-set 模型都会给这批相同传球打分。


In [ ]:
random_generator = np.random.default_rng(RANDOM_SEED)
women_game_ids = np.sort(women_matches["match_id"].unique())

WOMEN_TEST_GAMES = set(
    random_generator.choice(
        women_game_ids,
        size=int(len(women_game_ids) * WOMEN_TEST_SHARE),
        replace=False,
    )
)

print("women test games:", len(WOMEN_TEST_GAMES), "of", len(women_game_ids))
print("minimum passes per player:", MIN_PASSES_PER_PLAYER)


## 5. pass / shot 事件辅助函数

competition_events 返回的列有时是 flat columns（例如 pass_assisted_shot_id），有时在原始 JSON 里是 nested dict。这里的 helper 同时兼容两种形态。


In [ ]:
def is_missing_value(value):
    """Treat None/NaN/pandas missing scalars as missing, but leave lists/dicts alone."""
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    if missing is pd.NA:
        return True
    if isinstance(missing, (bool, np.bool_)):
        return bool(missing)
    return False


def statsbomb_name(value):
    """Return the StatsBomb object name, or the value itself if it is already flat."""
    if isinstance(value, dict):
        return value.get("name")
    return None if is_missing_value(value) else value


def statsbomb_id(value):
    """Return the StatsBomb object id, or the scalar itself if it is already an id."""
    if isinstance(value, dict):
        return value.get("id")
    return None if is_missing_value(value) else value


def first_available_value(row, column_names, default=None):
    """Return the first non-missing value among candidate dataframe columns."""
    for column_name in column_names:
        if column_name in row.index:
            value = row[column_name]
            if not is_missing_value(value):
                return value
    return default


def nested_event_value(row, object_column, nested_key, flat_columns, default=None):
    """Read a value from flat columns first, then from a nested event object."""
    value = first_available_value(row, flat_columns, None)
    if not is_missing_value(value):
        return value

    nested_object = first_available_value(row, [object_column], {})
    if isinstance(nested_object, dict):
        return nested_object.get(nested_key, default)
    return default


def pass_value(row, nested_key, flat_columns, default=None):
    """Read one pass attribute from flat StatsBomb columns or nested pass dict."""
    return nested_event_value(row, "pass", nested_key, flat_columns, default)


def shot_value(row, nested_key, flat_columns, default=None):
    """Read one shot attribute from flat StatsBomb columns or nested shot dict."""
    return nested_event_value(row, "shot", nested_key, flat_columns, default)


def parse_location_xy(location):
    """Parse a StatsBomb [x, y] location stored as list/tuple/array/string."""
    if isinstance(location, str) and location.strip().startswith("["):
        try:
            location = ast.literal_eval(location)
        except (SyntaxError, ValueError):
            return None

    if isinstance(location, (list, tuple, np.ndarray)) and len(location) >= 2:
        return float(location[0]), float(location[1])
    return None


def event_start_coordinates(row):
    """Extract pass start x/y in StatsBomb 120x80 coordinates."""
    parsed_location = parse_location_xy(first_available_value(row, ["location"]))
    if parsed_location is not None:
        return parsed_location

    return (
        first_available_value(row, ["location_x", "x", "start_x"], np.nan),
        first_available_value(row, ["location_y", "y", "start_y"], np.nan),
    )


def pass_end_coordinates(row):
    """Extract pass end x/y in StatsBomb 120x80 coordinates."""
    parsed_location = parse_location_xy(first_available_value(row, ["pass_end_location", "end_location"]))
    if parsed_location is not None:
        return parsed_location

    pass_object = first_available_value(row, ["pass"], {})
    if isinstance(pass_object, dict):
        parsed_location = parse_location_xy(pass_object.get("end_location"))
        if parsed_location is not None:
            return parsed_location

    return (
        first_available_value(row, ["pass_end_location_x", "end_x"], np.nan),
        first_available_value(row, ["pass_end_location_y", "end_y"], np.nan),
    )


def clean_numeric(value, default=np.nan):
    """Convert optional numeric value to float."""
    if is_missing_value(value):
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def distance_and_angle_from_goal_statsbomb(x_coordinate, y_coordinate):
    """Compute distance and visible goal angle from StatsBomb 120x80 coordinates."""
    if is_missing_value(x_coordinate) or is_missing_value(y_coordinate):
        return np.nan, np.nan

    spadl_x = float(x_coordinate) * 105 / 120
    spadl_y = float(y_coordinate) * 68 / 80
    dx = 105 - spadl_x
    dy = 34 - spadl_y

    distance = np.hypot(dx, dy)
    angle = abs(np.arctan2(7.32 * dx, dx**2 + dy**2 - (7.32 / 2) ** 2))
    return distance, angle


def boolean_flag(value):
    """Convert optional StatsBomb boolean fields to clean booleans."""
    return False if is_missing_value(value) else bool(value)


def pass_context(pass_type, play_pattern):
    """Collapse pass type + play pattern into broad chance-creation contexts."""
    pass_type = str(pass_type)
    play_pattern = str(play_pattern)

    if pass_type == "Corner" or play_pattern == "From Corner":
        return "Corner"
    if pass_type == "Free Kick" or play_pattern == "From Free Kick":
        return "Free Kick"
    if pass_type == "Throw-in" or play_pattern == "From Throw In":
        return "Throw-in"
    if pass_type in {"Goal Kick", "Kick Off"} or play_pattern in {"From Goal Kick", "From Kick Off"}:
        return "Restart"
    if play_pattern == "Regular Play" or pass_type in {"Regular Pass", "None", "nan"}:
        return "Open Play"
    return "Other"


def pass_end_zone(end_x, end_y):
    """Bucket the pass end location into broad attacking zones."""
    if is_missing_value(end_x) or is_missing_value(end_y):
        return "Unknown"

    end_x = float(end_x)
    end_y = float(end_y)

    if end_x >= 102 and 18 <= end_y <= 62:
        return "Penalty Box"
    if end_x >= 80:
        if end_y < 26.7:
            return "Final Third Left"
        if end_y > 53.3:
            return "Final Third Right"
        return "Final Third Central"
    if end_x >= 40:
        return "Middle Third"
    return "Defensive Third"


def shot_event_id(row):
    """Return a robust shot event id."""
    return first_available_value(row, ["id", "event_id"], None)


def shot_statsbomb_xg(row):
    """Extract StatsBomb xG from a shot event."""
    value = shot_value(row, "statsbomb_xg", ["shot_statsbomb_xg"], np.nan)
    return clean_numeric(value, np.nan)


def shot_key_pass_id(row):
    """Extract the key pass id from a shot event when present."""
    return shot_value(row, "key_pass_id", ["shot_key_pass_id"], None)


def build_shot_xg_lookups(shot_events):
    """Build shot-id -> xG and key-pass-id -> xG lookup tables."""
    shot_xg_by_id = {}
    key_pass_xg_by_id = {}

    for _, shot_event in shot_events.iterrows():
        event_id = shot_event_id(shot_event)
        statsbomb_xg = shot_statsbomb_xg(shot_event)
        if not is_missing_value(event_id) and not is_missing_value(statsbomb_xg):
            shot_xg_by_id[event_id] = float(statsbomb_xg)

        key_pass_id = shot_key_pass_id(shot_event)
        if not is_missing_value(key_pass_id) and not is_missing_value(statsbomb_xg):
            key_pass_xg_by_id[key_pass_id] = float(statsbomb_xg)

    return shot_xg_by_id, key_pass_xg_by_id


def event_to_xa_row(event, league_name, shot_xg_by_id, key_pass_xg_by_id):
    """Convert one StatsBomb pass event into one xA modeling row."""
    start_x, start_y = event_start_coordinates(event)
    end_x, end_y = pass_end_coordinates(event)

    start_x = clean_numeric(start_x)
    start_y = clean_numeric(start_y)
    end_x = clean_numeric(end_x)
    end_y = clean_numeric(end_y)

    pass_length = clean_numeric(pass_value(event, "length", ["pass_length"], np.nan))
    if is_missing_value(pass_length) and not any(is_missing_value(value) for value in [start_x, start_y, end_x, end_y]):
        pass_length = float(np.hypot(end_x - start_x, end_y - start_y))

    pass_angle = clean_numeric(pass_value(event, "angle", ["pass_angle"], np.nan))
    if is_missing_value(pass_angle) and not any(is_missing_value(value) for value in [start_x, start_y, end_x, end_y]):
        pass_angle = float(np.arctan2(end_y - start_y, end_x - start_x))

    start_distance, start_goal_angle = distance_and_angle_from_goal_statsbomb(start_x, start_y)
    end_distance, end_goal_angle = distance_and_angle_from_goal_statsbomb(end_x, end_y)

    play_pattern = statsbomb_name(first_available_value(event, ["play_pattern"], "Unknown")) or "Unknown"
    pass_type = statsbomb_name(pass_value(event, "type", ["pass_type"], "Regular Pass")) or "Regular Pass"
    pass_height = statsbomb_name(pass_value(event, "height", ["pass_height"], "Unknown")) or "Unknown"
    pass_body_part = statsbomb_name(pass_value(event, "body_part", ["pass_body_part"], "Unknown")) or "Unknown"
    pass_technique = statsbomb_name(pass_value(event, "technique", ["pass_technique"], "None")) or "None"
    pass_outcome = statsbomb_name(pass_value(event, "outcome", ["pass_outcome"], None)) or "Complete"

    pass_id = first_available_value(event, ["id", "event_id"], None)
    assisted_shot_id = pass_value(event, "assisted_shot_id", ["pass_assisted_shot_id"], None)

    target_xa = 0.0
    if not is_missing_value(assisted_shot_id) and assisted_shot_id in shot_xg_by_id:
        target_xa = shot_xg_by_id[assisted_shot_id]
    elif not is_missing_value(pass_id) and pass_id in key_pass_xg_by_id:
        target_xa = key_pass_xg_by_id[pass_id]

    target_xa = 0.0 if is_missing_value(target_xa) else float(target_xa)

    shot_assist_flag = boolean_flag(pass_value(event, "shot_assist", ["pass_shot_assist"], False))
    goal_assist_flag = boolean_flag(pass_value(event, "goal_assist", ["pass_goal_assist"], False))

    game_id_value = first_available_value(event, ["match_id", "game_id"], np.nan)
    player_value = first_available_value(event, ["player_id", "player"], None)

    return {
        "event_id": pass_id,
        "assisted_shot_id": assisted_shot_id,
        "game_id": int(game_id_value) if not is_missing_value(game_id_value) else np.nan,
        "player_id": statsbomb_id(player_value),
        "player_name": statsbomb_name(first_available_value(event, ["player_name", "player"], "Unknown")) or "Unknown",
        "team_name": statsbomb_name(first_available_value(event, ["team_name", "team"], "Unknown")) or "Unknown",
        "position_name": statsbomb_name(first_available_value(event, ["position_name", "position"], "Unknown")) or "Unknown",
        "league": league_name,
        "start_x": start_x,
        "start_y": start_y,
        "end_x": end_x,
        "end_y": end_y,
        "delta_x": end_x - start_x,
        "delta_y": end_y - start_y,
        "pass_length": pass_length,
        "pass_angle": pass_angle,
        "start_distance_to_goal": start_distance,
        "end_distance_to_goal": end_distance,
        "start_angle_to_goal": start_goal_angle,
        "end_angle_to_goal": end_goal_angle,
        "distance_reduction_to_goal": start_distance - end_distance,
        "pass_type": pass_type,
        "pass_height": pass_height,
        "pass_body_part": pass_body_part,
        "pass_technique": pass_technique,
        "pass_outcome": pass_outcome,
        "play_pattern": play_pattern,
        "pass_context": pass_context(pass_type, play_pattern),
        "end_zone": pass_end_zone(end_x, end_y),
        "under_pressure": boolean_flag(first_available_value(event, ["under_pressure"], False)),
        "is_cross": boolean_flag(pass_value(event, "cross", ["pass_cross"], False)),
        "is_cut_back": boolean_flag(pass_value(event, "cut_back", ["pass_cut_back"], False)),
        "is_switch": boolean_flag(pass_value(event, "switch", ["pass_switch"], False)),
        "is_through_ball": boolean_flag(pass_value(event, "through_ball", ["pass_through_ball"], False)),
        "is_inswinging": boolean_flag(pass_value(event, "inswinging", ["pass_inswinging"], False)),
        "is_outswinging": boolean_flag(pass_value(event, "outswinging", ["pass_outswinging"], False)),
        "end_in_box": bool(not is_missing_value(end_x) and not is_missing_value(end_y) and end_x >= 102 and 18 <= end_y <= 62),
        "end_in_six_yard_box": bool(not is_missing_value(end_x) and not is_missing_value(end_y) and end_x >= 114 and 30 <= end_y <= 50),
        "is_shot_assist": bool(target_xa > 0 or shot_assist_flag or goal_assist_flag),
        "is_goal_assist": goal_assist_flag,
        "assisted_shot_xg": target_xa,
    }


## 6. 加载传球样本并构造 xA label

每个 competition-season 会分别读取 pass events 和 shot events：pass 负责生成特征，shot 负责用 assisted_shot_id / shot_key_pass_id 找到对应的 shot_statsbomb_xg。


In [ ]:
def load_competition_event_xa_passes(competitions, cache_tag):
    """Load pass + shot events with competition_events and convert passes to xA rows."""
    passes_cache = CACHE_DIR / f"{cache_tag}_xa_passes_competition_events_method.pkl"

    if passes_cache.exists():
        print(f"[cache] {cache_tag} competition_events xA passes")
        return pd.read_pickle(passes_cache)

    pass_rows = []

    for competition in competitions:
        try:
            competition_passes = sb.competition_events(
                country=competition["country"],
                division=competition["division"],
                season=competition["season"],
                gender=competition["gender"],
                filters={"type": "Pass"},
            )
            competition_shots = sb.competition_events(
                country=competition["country"],
                division=competition["division"],
                season=competition["season"],
                gender=competition["gender"],
                filters={"type": "Shot"},
            )
        except Exception as error:
            print("  ! events", competition["league"], error)
            continue

        competition_passes = competition_passes.copy()
        competition_shots = competition_shots.copy()
        shot_xg_by_id, key_pass_xg_by_id = build_shot_xg_lookups(competition_shots)

        print(
            f"[load competition_events passes] {competition['league']}: "
            f"{len(competition_passes):,} passes | {len(competition_shots):,} shots"
        )

        league_rows = []
        for _, pass_event in competition_passes.iterrows():
            league_rows.append(
                event_to_xa_row(
                    pass_event,
                    competition["league"],
                    shot_xg_by_id,
                    key_pass_xg_by_id,
                )
            )

        league_frame = pd.DataFrame(league_rows)
        print(
            "   shot-assist passes with xA label:",
            int((league_frame["assisted_shot_xg"] > 0).sum()),
            "| label sum:",
            round(float(league_frame["assisted_shot_xg"].sum()), 2),
        )
        pass_rows.extend(league_rows)

    passes = pd.DataFrame(pass_rows)
    if passes.empty:
        raise ValueError("No pass rows were loaded. Check StatsBomb competition filters and network/cache availability.")

    passes = passes.dropna(subset=["game_id", "player_id"]).copy()
    passes["assisted_shot_xg"] = pd.to_numeric(passes["assisted_shot_xg"], errors="coerce").fillna(0.0)
    passes.to_pickle(passes_cache)
    print(f"[cache] wrote {cache_tag} competition_events xA passes: {len(passes):,} passes")
    return passes


women_passes = load_competition_event_xa_passes(WOMEN_COMPETITIONS, "women")
men_passes = load_competition_event_xa_passes(MEN_COMPETITIONS, "men2015")

women_train_passes = women_passes[~women_passes.game_id.isin(WOMEN_TEST_GAMES)].copy()
women_test_passes = women_passes[women_passes.game_id.isin(WOMEN_TEST_GAMES)].copy()

print(
    f"xA pass samples | men train={len(men_passes):,} "
    f"women train={len(women_train_passes):,} women test={len(women_test_passes):,}"
)
print(
    f"shot-assist labels | men={int((men_passes.assisted_shot_xg > 0).sum()):,} "
    f"women train={int((women_train_passes.assisted_shot_xg > 0).sum()):,} "
    f"women test={int((women_test_passes.assisted_shot_xg > 0).sum()):,}"
)


## 7. input 数据预处理

这里从男足训练集 + 女足训练集构建一套共享 one-hot feature schema，然后让三个数据集使用完全相同的 feature columns。


In [ ]:
NUMERIC_XA_FEATURES = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "delta_x",
    "delta_y",
    "pass_length",
    "pass_angle",
    "start_distance_to_goal",
    "end_distance_to_goal",
    "start_angle_to_goal",
    "end_angle_to_goal",
    "distance_reduction_to_goal",
]

BOOLEAN_XA_FEATURES = [
    "under_pressure",
    "is_cross",
    "is_cut_back",
    "is_switch",
    "is_through_ball",
    "is_inswinging",
    "is_outswinging",
    "end_in_box",
    "end_in_six_yard_box",
]

CATEGORICAL_XA_FEATURES = [
    "pass_type",
    "pass_height",
    "pass_body_part",
    "pass_technique",
    "pass_outcome",
    "play_pattern",
    "pass_context",
    "end_zone",
]

XA_BASE_FEATURES = NUMERIC_XA_FEATURES + BOOLEAN_XA_FEATURES + CATEGORICAL_XA_FEATURES


def build_xa_design_matrix(passes, expected_columns=None):
    """Turn pass rows into the numeric matrix expected by XGBoost."""
    features = passes[XA_BASE_FEATURES].copy()

    for column in NUMERIC_XA_FEATURES:
        features[column] = pd.to_numeric(features[column], errors="coerce").fillna(0.0)
    for column in BOOLEAN_XA_FEATURES:
        features[column] = features[column].fillna(False).astype(int)
    for column in CATEGORICAL_XA_FEATURES:
        features[column] = features[column].fillna("Unknown").astype(str)

    features = pd.get_dummies(features, columns=CATEGORICAL_XA_FEATURES, dtype=float)
    if expected_columns is not None:
        features = features.reindex(columns=expected_columns, fill_value=0.0)
    return features


training_passes_for_feature_schema = pd.concat(
    [men_passes, women_train_passes],
    ignore_index=True,
)
xa_feature_columns = build_xa_design_matrix(training_passes_for_feature_schema).columns

men_train_features = build_xa_design_matrix(men_passes, expected_columns=xa_feature_columns)
women_train_features = build_xa_design_matrix(women_train_passes, expected_columns=xa_feature_columns)
women_test_features = build_xa_design_matrix(women_test_passes, expected_columns=xa_feature_columns)

print("xA base features:", XA_BASE_FEATURES)
print("shared one-hot model columns:", len(xa_feature_columns))
print(
    "feature matrix shapes | "
    f"men_train={men_train_features.shape} "
    f"women_train={women_train_features.shape} "
    f"women_test={women_test_features.shape}"
)


## 8. 训练 xA 模型并给同一批女足 test passes 打分

和 xG notebook 一样训练三个模型：男足 full set、男足 downsampled、女足。主比较使用 downsampled men vs women。


In [ ]:
downsample_size = len(women_train_passes)
if len(men_passes) < downsample_size:
    raise ValueError("Men pass sample is smaller than women train sample; cannot downsample as planned.")

men_downsampled_passes = men_passes.sample(
    n=downsample_size,
    random_state=RANDOM_SEED,
    replace=False,
).copy()

men_downsampled_features = build_xa_design_matrix(
    men_downsampled_passes,
    expected_columns=xa_feature_columns,
)


def train_xa_model(features, target, model_label):
    """Train one pass-level expected-assist regressor."""
    model = XGBRegressor(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.90,
        colsample_bytree=0.90,
        objective="reg:squarederror",
        random_state=RANDOM_SEED,
        n_jobs=4,
    )
    model.fit(features, target)
    print(f"[trained] {model_label}: rows={len(features):,} target_sum={target.sum():.2f}")
    return model


def predict_xa(model, features):
    """Predict xA and clip to the natural [0, 1] xG scale."""
    return np.clip(model.predict(features), 0.0, 1.0)


men_full_xa_model = train_xa_model(
    men_train_features,
    men_passes["assisted_shot_xg"],
    "men full xA",
)
men_downsampled_xa_model = train_xa_model(
    men_downsampled_features,
    men_downsampled_passes["assisted_shot_xg"],
    "men downsampled xA",
)
women_xa_model = train_xa_model(
    women_train_features,
    women_train_passes["assisted_shot_xg"],
    "women xA",
)

xa_predictions = women_test_passes.copy()
xa_predictions["men_full_model_xa"] = predict_xa(men_full_xa_model, women_test_features)
xa_predictions["men_downsampled_model_xa"] = predict_xa(men_downsampled_xa_model, women_test_features)
xa_predictions["women_model_xa"] = predict_xa(women_xa_model, women_test_features)
xa_predictions["xa_delta"] = xa_predictions["men_downsampled_model_xa"] - xa_predictions["women_model_xa"]

print("Training pass counts:")
print(f"- men full: {len(men_passes):,}")
print(f"- men downsampled: {len(men_downsampled_passes):,}")
print(f"- women train: {len(women_train_passes):,}")
print(f"- women test: {len(women_test_passes):,}")


## 9. 球员位置、门将过滤和 Spearman helper

优先使用已有 position cache；如果没有，就从本 notebook 的女足 pass rows 里抽取球员位置。


In [ ]:
POSITION_CACHE_CANDIDATES = [
    CACHE_DIR / "women_positions_competition_events_method.pkl",
    CACHE_DIR / "women_positions_statsbombpy_remote.pkl",
    CACHE_DIR / "women_pos.parquet",
]


def load_women_position_rows(fallback_passes=None):
    """Load cached women player-position observations, or use pass-row fallback."""
    for position_cache in POSITION_CACHE_CANDIDATES:
        if position_cache.exists():
            print(f"[cache] women positions: {position_cache}")
            if position_cache.suffix == ".parquet":
                positions = pd.read_parquet(position_cache).copy()
            else:
                positions = pd.read_pickle(position_cache).copy()
            break
    else:
        if fallback_passes is None:
            raise FileNotFoundError("No women position cache found and no fallback pass rows were provided.")
        print("[fallback] women positions from xA pass rows")
        positions = fallback_passes[["player_id", "position_name"]].copy()

    required_columns = {"player_id", "position_name"}
    missing_columns = required_columns.difference(positions.columns)
    if missing_columns:
        raise ValueError(f"Position cache missing columns: {sorted(missing_columns)}")

    positions = positions[list(required_columns)].dropna().copy()
    positions["player_id_key"] = pd.to_numeric(positions["player_id"], errors="coerce").astype("Int64")
    positions["position_name"] = positions["position_name"].astype(str)
    positions = positions.dropna(subset=["player_id_key", "position_name"])
    return positions


def build_primary_position_table(position_rows):
    """Choose the most frequent observed position for each player."""
    position_counts = (
        position_rows.groupby(["player_id_key", "position_name"])
        .size()
        .reset_index(name="position_observations")
        .sort_values(["player_id_key", "position_observations", "position_name"], ascending=[True, False, True])
    )

    return (
        position_counts.groupby("player_id_key", as_index=False)
        .first()
        .rename(columns={"position_name": "primary_position"})
    )


def bucket_position(position_name):
    """Map granular StatsBomb positions to broad tactical buckets."""
    position_name = str(position_name)

    if "Goalkeeper" in position_name:
        return "Goalkeeper"
    if "Center Back" in position_name:
        return "Center Back"
    if "Wing Back" in position_name or position_name in {"Left Back", "Right Back"}:
        return "Fullback / Wing Back"
    if "Defensive Midfield" in position_name:
        return "Defensive Midfield"
    if "Center Midfield" in position_name:
        return "Central Midfield"
    if "Attacking Midfield" in position_name:
        return "Attacking Midfield"
    if position_name in {"Left Midfield", "Right Midfield", "Left Wing", "Right Wing"}:
        return "Wide Midfielder / Winger"
    if "Center Forward" in position_name or position_name == "Striker":
        return "Forward"
    return "Other / Unknown"


def spearman_if_available(left, right, min_observations=2):
    """Return Spearman correlation when there are enough non-constant observations."""
    paired_values = pd.DataFrame({"left": left, "right": right}).dropna()
    if len(paired_values) < min_observations:
        return np.nan
    if paired_values["left"].nunique(dropna=True) < 2 or paired_values["right"].nunique(dropna=True) < 2:
        return np.nan
    return stats.spearmanr(paired_values["left"], paired_values["right"])[0]


women_position_rows = load_women_position_rows(women_passes)
primary_positions = build_primary_position_table(women_position_rows)
primary_positions["position_bucket"] = primary_positions["primary_position"].map(bucket_position)
goalkeeper_player_id_keys = set(
    primary_positions.loc[primary_positions["position_bucket"].eq("Goalkeeper"), "player_id_key"]
)

print("primary-position rows:", len(primary_positions))
print("goalkeepers identified:", len(goalkeeper_player_id_keys))


## 10. 球员层面汇总：男足 xA vs 女足 xA

按球员把 pass-level xA 加总，再比较两个模型下的球员排名。这里排除门将，并要求测试集传球数达到阈值。


In [ ]:
player_xa_comparison_all = (
    xa_predictions.groupby("player_id")
    .agg(
        player_name=("player_name", "first"),
        men_downsampled_xa=("men_downsampled_model_xa", "sum"),
        women_xa=("women_model_xa", "sum"),
        observed_xa=("assisted_shot_xg", "sum"),
        pass_count=("women_model_xa", "size"),
        shot_assist_count=("is_shot_assist", "sum"),
        goal_assist_count=("is_goal_assist", "sum"),
    )
    .reset_index()
)

player_xa_comparison_all["player_id_key"] = pd.to_numeric(
    player_xa_comparison_all["player_id"],
    errors="coerce",
).astype("Int64")

player_xa_comparison_all = player_xa_comparison_all.merge(
    primary_positions,
    on="player_id_key",
    how="left",
)
player_xa_comparison_all["primary_position"] = player_xa_comparison_all["primary_position"].fillna("Unknown")
player_xa_comparison_all["position_bucket"] = player_xa_comparison_all["position_bucket"].fillna("Other / Unknown")
player_xa_comparison_all["xa_delta"] = (
    player_xa_comparison_all["men_downsampled_xa"] - player_xa_comparison_all["women_xa"]
)
player_xa_comparison_all["abs_xa_delta"] = player_xa_comparison_all["xa_delta"].abs()

player_xa_comparison = player_xa_comparison_all[
    (player_xa_comparison_all["pass_count"] >= MIN_PASSES_PER_PLAYER)
    & (~player_xa_comparison_all["player_id_key"].isin(goalkeeper_player_id_keys))
].copy()

rho_xa = spearman_if_available(
    player_xa_comparison["men_downsampled_xa"],
    player_xa_comparison["women_xa"],
)

print(f"xA | players={len(player_xa_comparison)} | Spearman downsampled men vs women={rho_xa:.3f}")
print("Total predicted xA on women test passes:")
print(f"- men downsampled model: {xa_predictions['men_downsampled_model_xa'].sum():.2f}")
print(f"- women model: {xa_predictions['women_model_xa'].sum():.2f}")
print(f"- realized assisted-shot xG label: {xa_predictions['assisted_shot_xg'].sum():.2f}")

display(
    player_xa_comparison.sort_values("abs_xa_delta", ascending=False)[
        [
            "player_id",
            "player_name",
            "primary_position",
            "position_bucket",
            "pass_count",
            "shot_assist_count",
            "observed_xa",
            "men_downsampled_xa",
            "women_xa",
            "xa_delta",
        ]
    ].head(20).round(4)
)


## 11. 球员 scatter：同一批女足 test passes，不同训练数据模型

点越贴近对角线，说明男足训练模型和女足训练模型对球员 xA 总量 / 排名越接近。


In [ ]:
if player_xa_comparison.empty:
    print("No players passed the filtering threshold; lower MIN_PASSES_PER_PLAYER if needed.")
else:
    fig, axis = plt.subplots(figsize=(7, 7))

    axis.scatter(
        player_xa_comparison["women_xa"],
        player_xa_comparison["men_downsampled_xa"],
        alpha=0.70,
        s=34,
    )

    axis_min = min(player_xa_comparison["women_xa"].min(), player_xa_comparison["men_downsampled_xa"].min())
    axis_max = max(player_xa_comparison["women_xa"].max(), player_xa_comparison["men_downsampled_xa"].max())
    padding = (axis_max - axis_min) * 0.05 if axis_max > axis_min else 0.1
    axis.plot(
        [axis_min - padding, axis_max + padding],
        [axis_min - padding, axis_max + padding],
        linestyle="--",
        color="black",
        linewidth=1,
    )

    for _, row in player_xa_comparison.sort_values("abs_xa_delta", ascending=False).head(8).iterrows():
        label = row["player_name"] if row["player_name"] != "Unknown" else str(row["player_id"])
        axis.annotate(
            label,
            (row["women_xa"], row["men_downsampled_xa"]),
            fontsize=8,
            alpha=0.8,
            xytext=(4, 4),
            textcoords="offset points",
        )

    axis.set_xlabel("Women-trained model xA")
    axis.set_ylabel("Downsampled men-trained model xA")
    axis.set_title(f"Player xA comparison | Spearman={rho_xa:.3f}")
    axis.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


## 12. 模型预测和 realized xA label 的关系

assisted_shot_xg 是稀疏 realized label，不是真正的 ground-truth expectation；所以这里主要看整体尺度、相关性和误差，避免把单脚传球层面的误差过度解读。


In [ ]:
observed_xa = xa_predictions["assisted_shot_xg"].astype(float)
shot_assist_mask = observed_xa > 0

comparison_rows = []
for source_label, prediction_column in [
    ("men_full_model_xa", "men_full_model_xa"),
    ("men_downsampled_model_xa", "men_downsampled_model_xa"),
    ("women_model_xa", "women_model_xa"),
]:
    prediction = xa_predictions[prediction_column].astype(float)
    comparison_rows.append(
        {
            "source": source_label,
            "passes": len(prediction),
            "shot_assist_passes": int(shot_assist_mask.sum()),
            "total_predicted_xa": prediction.sum(),
            "total_realized_xa": observed_xa.sum(),
            "mean_predicted_xa": prediction.mean(),
            "mae_vs_realized_xa": np.mean(np.abs(prediction - observed_xa)),
            "rmse_vs_realized_xa": np.sqrt(np.mean((prediction - observed_xa) ** 2)),
            "mae_on_shot_assists": (
                np.mean(np.abs(prediction[shot_assist_mask] - observed_xa[shot_assist_mask]))
                if shot_assist_mask.any()
                else np.nan
            ),
            "pearson_vs_realized_xa": prediction.corr(observed_xa, method="pearson"),
            "spearman_vs_realized_xa": prediction.corr(observed_xa, method="spearman"),
        }
    )

model_vs_realized_xa = pd.DataFrame(comparison_rows)
display(model_vs_realized_xa.round(5))

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
scatter_specs = [
    (
        "assisted_shot_xg",
        "men_downsampled_model_xa",
        "Realized assisted-shot xG",
        "Downsampled men-trained model xA",
        "Downsampled men model vs realized xA",
    ),
    (
        "assisted_shot_xg",
        "women_model_xa",
        "Realized assisted-shot xG",
        "Women-trained model xA",
        "Women-trained model vs realized xA",
    ),
    (
        "women_model_xa",
        "men_downsampled_model_xa",
        "Women-trained model xA",
        "Downsampled men-trained model xA",
        "Downsampled men model vs women-trained model",
    ),
]

max_xa = xa_predictions[["assisted_shot_xg", "men_downsampled_model_xa", "women_model_xa"]].max().max()
axis_limit = min(1.0, max(0.02, max_xa * 1.05))

for axis, (x_column, y_column, x_label, y_label, title) in zip(axes, scatter_specs):
    axis.scatter(
        xa_predictions[x_column],
        xa_predictions[y_column],
        alpha=0.20,
        s=8,
    )
    axis.plot([0, axis_limit], [0, axis_limit], linestyle="--", color="black", linewidth=1)
    axis.set_title(title)
    axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.set_xlim(0, axis_limit)
    axis.set_ylim(0, axis_limit)

plt.tight_layout()
plt.show()


## 13. 补充诊断：男足 full set 和 downsized 的差异

这一步不是主比较，只是检查“男足训练样本更多”这件事本身会不会主导预测差异。


In [ ]:
training_size_comparison_rows = []
comparison_pairs = [
    ("men_full_vs_men_downsampled", "men_full_model_xa", "men_downsampled_model_xa"),
    ("men_full_vs_women", "men_full_model_xa", "women_model_xa"),
    ("men_downsampled_vs_women", "men_downsampled_model_xa", "women_model_xa"),
]

for label, left_column, right_column in comparison_pairs:
    left = xa_predictions[left_column]
    right = xa_predictions[right_column]
    training_size_comparison_rows.append(
        {
            "comparison": label,
            "left_column": left_column,
            "right_column": right_column,
            "mae_between_predictions": np.mean(np.abs(left - right)),
            "rmse_between_predictions": np.sqrt(np.mean((left - right) ** 2)),
            "pearson_between_predictions": left.corr(right, method="pearson"),
            "spearman_between_predictions": left.corr(right, method="spearman"),
            "left_total_xa": left.sum(),
            "right_total_xa": right.sum(),
        }
    )

training_size_comparison = pd.DataFrame(training_size_comparison_rows)
display(training_size_comparison.round(5))

fig, axis = plt.subplots(figsize=(6, 5))
max_xa = xa_predictions[["men_full_model_xa", "men_downsampled_model_xa"]].max().max()
axis_limit = min(1.0, max(0.02, max_xa * 1.05))

axis.scatter(
    xa_predictions["men_downsampled_model_xa"],
    xa_predictions["men_full_model_xa"],
    alpha=0.20,
    s=8,
)
axis.plot([0, axis_limit], [0, axis_limit], linestyle="--", color="black", linewidth=1)
axis.set_title("Full men model vs downsampled men model")
axis.set_xlabel("Downsampled men-trained model xA")
axis.set_ylabel("Full men-trained model xA")
axis.set_xlim(0, axis_limit)
axis.set_ylim(0, axis_limit)

plt.tight_layout()
plt.show()


## 14. 按传球终点 grid 比较模型差异

这里把女足测试集传球按照终点位置落到 12x16 grid，观察哪些区域的 men_downsampled_model_xa - women_model_xa 更大。


In [ ]:
XA_GRID_ROWS = 12
XA_GRID_COLS = 16
MIN_GRID_PASSES = 200
STATSBOMB_PITCH_LENGTH = 120.0
STATSBOMB_PITCH_WIDTH = 80.0


def add_xa_end_grid_columns(passes, rows=XA_GRID_ROWS, cols=XA_GRID_COLS):
    """Assign each pass to a grid cell based on end location."""
    passes = passes.copy()
    valid_coordinates = passes[["end_x", "end_y"]].notna().all(axis=1)
    passes = passes[valid_coordinates].copy()

    passes["grid_col"] = np.floor(passes["end_x"] / STATSBOMB_PITCH_LENGTH * cols).astype(int).clip(0, cols - 1)
    passes["grid_row"] = np.floor(passes["end_y"] / STATSBOMB_PITCH_WIDTH * rows).astype(int).clip(0, rows - 1)
    return passes


def summarize_xa_delta_by_end_grid(passes, rows=XA_GRID_ROWS, cols=XA_GRID_COLS):
    """Summarize pass-level xA deltas by pass end-location grid."""
    passes_with_grid = add_xa_end_grid_columns(passes, rows=rows, cols=cols)
    passes_with_grid["xa_delta"] = passes_with_grid["men_downsampled_model_xa"] - passes_with_grid["women_model_xa"]

    all_cells = pd.MultiIndex.from_product(
        [range(rows), range(cols)],
        names=["grid_row", "grid_col"],
    ).to_frame(index=False)

    grid_summary = (
        passes_with_grid.groupby(["grid_row", "grid_col"])
        .agg(
            pass_count=("xa_delta", "size"),
            observed_xa=("assisted_shot_xg", "sum"),
            men_downsampled_xa=("men_downsampled_model_xa", "sum"),
            women_xa=("women_model_xa", "sum"),
            aggregate_xa_delta=("xa_delta", "sum"),
            mean_xa_delta=("xa_delta", "mean"),
            median_xa_delta=("xa_delta", "median"),
            mean_women_xa=("women_model_xa", "mean"),
            mean_men_downsampled_xa=("men_downsampled_model_xa", "mean"),
        )
        .reset_index()
    )

    grid_summary = all_cells.merge(grid_summary, on=["grid_row", "grid_col"], how="left")
    count_columns = ["pass_count"]
    sum_columns = ["observed_xa", "men_downsampled_xa", "women_xa", "aggregate_xa_delta"]
    grid_summary[count_columns + sum_columns] = grid_summary[count_columns + sum_columns].fillna(0)
    return passes_with_grid, grid_summary


xa_grid_predictions, xa_grid_delta_summary = summarize_xa_delta_by_end_grid(xa_predictions)
valid_grid_mask = xa_grid_delta_summary["pass_count"] >= MIN_GRID_PASSES

print("Grid delta definition: men_downsampled_model_xa - women_model_xa")
print(f"women test passes assigned to grid: {len(xa_grid_predictions):,} of {len(xa_predictions):,}")
print(f"non-empty grid cells: {(xa_grid_delta_summary.pass_count > 0).sum()} of {len(xa_grid_delta_summary)}")
print(f"grid cells with at least {MIN_GRID_PASSES} passes: {valid_grid_mask.sum()} of {len(xa_grid_delta_summary)}")

print("\nLargest absolute aggregate deltas by pass end grid:")
display(
    xa_grid_delta_summary[valid_grid_mask]
    .assign(abs_aggregate_delta=lambda frame: frame["aggregate_xa_delta"].abs())
    .sort_values("abs_aggregate_delta", ascending=False)
    .head(20)
    .round(5)
)


## 14.1 传球终点 grid visualization

左图看女足测试集传球终点分布，右图看每个终点区域的平均模型差异。


In [ ]:
def grid_metric_matrix(grid_summary, metric, rows=XA_GRID_ROWS, cols=XA_GRID_COLS, min_passes=None):
    """Convert long grid summary into a rows x cols matrix."""
    matrix = np.full((rows, cols), np.nan)
    for _, row in grid_summary.iterrows():
        if min_passes is not None and row["pass_count"] < min_passes:
            continue
        matrix[int(row["grid_row"]), int(row["grid_col"])] = row[metric]
    return matrix


def plot_xa_grid_heatmap(axis, grid_summary, metric, title, cmap="viridis", center_zero=False, min_passes=None):
    """Draw one pitch heatmap from a grid metric."""
    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="black", linewidth=1)
    pitch.draw(ax=axis)

    matrix = grid_metric_matrix(grid_summary, metric, min_passes=min_passes)
    if np.all(np.isnan(matrix)):
        axis.set_title(title + " (no valid cells)")
        return None

    if center_zero:
        value_limit = np.nanmax(np.abs(matrix))
        value_limit = value_limit if value_limit > 0 else 1e-6
        vmin, vmax = -value_limit, value_limit
    else:
        vmin, vmax = np.nanmin(matrix), np.nanmax(matrix)

    image = axis.imshow(
        matrix,
        extent=(0, STATSBOMB_PITCH_LENGTH, STATSBOMB_PITCH_WIDTH, 0),
        origin="upper",
        cmap=cmap,
        alpha=0.72,
        vmin=vmin,
        vmax=vmax,
    )
    axis.set_title(title)
    return image


fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))
image_0 = plot_xa_grid_heatmap(
    axes[0],
    xa_grid_delta_summary,
    "pass_count",
    "Women test pass end-count by grid",
    cmap="viridis",
)
image_1 = plot_xa_grid_heatmap(
    axes[1],
    xa_grid_delta_summary,
    "mean_xa_delta",
    f"Mean xA delta by end grid | min {MIN_GRID_PASSES} passes",
    cmap="coolwarm",
    center_zero=True,
    min_passes=MIN_GRID_PASSES,
)

for axis, image in zip(axes, [image_0, image_1]):
    if image is not None:
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


## 15. 按 pass context / end zone 分析模型差异

xA 全部来自传球，所以这里不按 action type，而是按传球上下文、终点区域、传球高度等 pass profile 来看差异。


In [ ]:
def summarize_xa_by_group(group_column, min_passes=MIN_PASSES_PER_PASS_GROUP):
    """Summarize xA model differences for one categorical pass grouping."""
    summary = (
        xa_predictions.groupby(group_column)
        .agg(
            passes=("xa_delta", "size"),
            shot_assist_passes=("is_shot_assist", "sum"),
            observed_xa=("assisted_shot_xg", "sum"),
            men_downsampled_xa=("men_downsampled_model_xa", "sum"),
            women_xa=("women_model_xa", "sum"),
            aggregate_xa_delta=("xa_delta", "sum"),
            mean_xa_delta=("xa_delta", "mean"),
            median_xa_delta=("xa_delta", "median"),
            mean_men_downsampled_xa=("men_downsampled_model_xa", "mean"),
            mean_women_xa=("women_model_xa", "mean"),
        )
        .reset_index()
    )
    summary = summary[summary["passes"] >= min_passes].copy()
    return summary.sort_values("mean_xa_delta")


group_summaries = {}
for group_column in ["pass_context", "end_zone", "pass_height", "pass_outcome"]:
    summary = summarize_xa_by_group(group_column)
    group_summaries[group_column] = summary
    print(f"\n{group_column} groups with at least {MIN_PASSES_PER_PASS_GROUP:,} passes:")
    display(summary.round(5))


def plot_group_delta(axis, summary, group_column, title):
    """Horizontal bar chart of mean xA delta for one grouping."""
    if summary.empty:
        axis.set_title(title + " (no groups)")
        return

    colors = np.where(summary["mean_xa_delta"] >= 0, "#D55E00", "#0072B2")
    axis.barh(summary[group_column], summary["mean_xa_delta"], color=colors, alpha=0.75)
    axis.axvline(0, color="black", linewidth=1)
    axis.set_xlabel("Mean xA delta: men model - women model")
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.20)


fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_group_delta(axes[0], group_summaries["pass_context"], "pass_context", "By pass context")
plot_group_delta(axes[1], group_summaries["end_zone"], "end_zone", "By pass end zone")
plt.tight_layout()
plt.show()


## 16. 按 position bucket 分析 xA 排名差异

这部分和 VAEP / xG notebook 的 position-bucket 分析保持一致：看每个位置组内的 Spearman、总 xA delta 和组内排名变化。


In [ ]:
player_xa_by_position = player_xa_comparison.copy()

if player_xa_by_position.empty:
    print("No players available for position-bucket summary after filtering.")
else:
    player_xa_by_position["bucket_men_rank"] = player_xa_by_position.groupby("position_bucket")[
        "men_downsampled_xa"
    ].rank(ascending=False, method="min")
    player_xa_by_position["bucket_women_rank"] = player_xa_by_position.groupby("position_bucket")[
        "women_xa"
    ].rank(ascending=False, method="min")
    player_xa_by_position["bucket_rank_delta"] = (
        player_xa_by_position["bucket_men_rank"] - player_xa_by_position["bucket_women_rank"]
    )
    player_xa_by_position["abs_bucket_rank_delta"] = player_xa_by_position["bucket_rank_delta"].abs()

    bucket_summary_rows = []
    for position_bucket, bucket_players in player_xa_by_position.groupby("position_bucket"):
        bucket_summary_rows.append(
            {
                "position_bucket": position_bucket,
                "players": len(bucket_players),
                "passes": int(bucket_players["pass_count"].sum()),
                "shot_assist_passes": int(bucket_players["shot_assist_count"].sum()),
                "observed_xa": bucket_players["observed_xa"].sum(),
                "spearman_men_vs_women_xa": spearman_if_available(
                    bucket_players["men_downsampled_xa"],
                    bucket_players["women_xa"],
                    min_observations=MIN_PLAYERS_FOR_BUCKET_SPEARMAN,
                ),
                "aggregate_xa_delta": bucket_players["xa_delta"].sum(),
                "mean_xa_delta": bucket_players["xa_delta"].mean(),
                "median_xa_delta": bucket_players["xa_delta"].median(),
                "mean_abs_xa_delta": bucket_players["abs_xa_delta"].mean(),
                "mean_bucket_rank_delta": bucket_players["bucket_rank_delta"].mean(),
                "median_abs_bucket_rank_delta": bucket_players["abs_bucket_rank_delta"].median(),
                "max_abs_bucket_rank_delta": bucket_players["abs_bucket_rank_delta"].max(),
            }
        )

    position_bucket_xa_summary = (
        pd.DataFrame(bucket_summary_rows)
        .sort_values(["players", "passes"], ascending=False)
        .reset_index(drop=True)
    )

    print("Position-bucket summary for women test-set players with at least", MIN_PASSES_PER_PLAYER, "passes:")
    display(position_bucket_xa_summary.round(4))

    print("\nPlayers with largest within-bucket rank changes:")
    display(
        player_xa_by_position.sort_values("abs_bucket_rank_delta", ascending=False)[
            [
                "player_id",
                "player_name",
                "primary_position",
                "position_bucket",
                "pass_count",
                "observed_xa",
                "men_downsampled_xa",
                "women_xa",
                "xa_delta",
                "bucket_men_rank",
                "bucket_women_rank",
                "bucket_rank_delta",
            ]
        ]
        .head(25)
        .round(4)
    )


## 17. Read-out

这个 notebook 最后应该回答：

1. 男足训练的 xA 模型和女足训练的 xA 模型，在同一批女足 test passes 上给出的球员 xA 排名有多一致？
2. 两个模型的差异是否主要来自训练数据量（full men vs downsampled men），还是来自男女足 pass-to-shot-xG 关系本身？
3. 差异是否集中在某些传球终点区域、传球上下文或 position bucket？

如果 xA 的 Spearman 接近 xG / xT，说明 chance-creation pass model 在跨性别迁移上比较稳定；如果明显低于 xG / xT 或在某些 pass profile 上偏差集中，就说明助攻前传球的价值估计比 shot-only / location-only 指标更容易受训练数据分布影响。
